In [1]:
import pandas as pd
import numpy as np

import datetime as dt
from pandas.tseries.offsets import *

import yfinance as yf

In [2]:
import json

# Data Preprocessing

Load historical finance data for United States Oil Fund /USO/.

Load the list of the tracked months in grocery data from json file and check the type of the returned data.

In [3]:
with open("../globals//month_list.json", "r") as file:
    months_list = json.load(file) 

months_list, type(months_list)

(['2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03'], list)

Load the collection date of grocery data from json file and check the type of the returned data.

In [4]:
with open("../globals/date_collection.json", "r") as file:
    date_collection = json.load(file) 

date_collection, type(date_collection)

(['2026-03-20'], list)

We will load USO data for the same period as grocery data has been collected.

We will offset the collection date with one day, because the historical data is returned up to 23:59:59 on the day before the date specified.

In [5]:
# start date
start_day = '-01'
start = pd.to_datetime(months_list[0] + start_day)

# end date
end = pd.to_datetime(date_collection[0]) + DateOffset(days=1)

Verify "start" and "end" variables.

In [6]:
start, end

(Timestamp('2025-10-01 00:00:00'), Timestamp('2026-03-21 00:00:00'))

In [7]:
# declare a ticker for United States Oil Fund, LP (USO)
ticker = "USO"

# download data from yfinance
finance_data = yf.download(ticker, start, end)

# show latest 5 samples
finance_data.tail(5)

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,USO,USO,USO,USO,USO
Date,,,,,
2026-03-16,115.029999,118.910004,114.360001,116.930000,66869700
2026-03-17,118.839996,119.129997,116.070000,117.419998,40127600
2026-03-18,121.669998,122.870003,117.449997,121.029999,70647800
2026-03-19,117.360001,125.190002,114.680000,120.400002,96157300
2026-03-20,121.430000,123.019997,118.330002,118.959999,49738900


The end date is correct.

Analyze the multiindex finance dataframe.

In [8]:
# the size of data frame
finance_data.shape

(118, 5)

The DataFrame has 118 rows - one per each working day and 5 columns for 5 price types.

Check the number and the names of the levels.

In [9]:
finance_data.columns.nlevels, finance_data.columns.names

(2, FrozenList(['Price', 'Ticker']))

The data has 2 levels with names "Price" and "Ticker".

Get the unique types of "Price" level.

In [10]:
finance_data.columns.get_level_values(0).unique()

Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')

"Price" level has 5 types - 'Close', 'High', 'Low', 'Open', 'Volume'.

For our study we will use only "Close" price.

In [11]:
# create a table with only "Close" prices for USO fund.
close_prices = finance_data.xs('Close', level=0, axis=1)
close_prices

Ticker,USO
Date,
2025-10-01,73.139999
2025-10-02,71.540001
2025-10-03,71.709999
2025-10-06,72.870003
2025-10-07,73.199997
...,...
2026-03-16,115.029999
2026-03-17,118.839996
2026-03-18,121.669998


Since in grocery dataset we have monthly average prices and in finance dataset we have closing prices for each working day, we will calculate average monhtly closing price for USO fund.  

In [ ]:
# init finance dataset with average closing price
mean_col_names = ['Date', 'USO']
finance_data_mean = pd.DataFrame(columns=mean_col_names)
finance_data_mean

,Date,USO


In [13]:
# a function that returns a new created row {Date: input month, USO: average monthly price}
def mean_by_month(month):
  # calculate average closing price for the month
  temp_close_prices_mean = close_prices.loc[month].mean()

  # create new row
  new_row = pd.DataFrame({mean_col_names[0]: month, mean_col_names[1]: temp_close_prices_mean.iloc[0]}, index=[0])
  return new_row

In [14]:
# append a new row to finance_data_mean for each month in months_list 

rows_list = []

for m in months_list:
  row = mean_by_month(m)
  rows_list.append(row)

finance_data_mean = pd.concat(rows_list, ignore_index = True)

Show the table with average monthlly closing price.

In [15]:
finance_data_mean

,Date,USO
0,2025-10,71.118695
1,2025-11,70.956316
2,2025-12,69.484546
3,2026-01,72.635500
4,2026-02,78.535789
5,2026-03,108.325333


Set "Date" column as index.

In [16]:
finance_data_mean.set_index('Date', inplace=True)
finance_data_mean

,USO
Date,
2025-10,71.118695
2025-11,70.956316
2025-12,69.484546
2026-01,72.635500
2026-02,78.535789
2026-03,108.325333


Round the price value to two decimals.

In [17]:
finance_data_mean.USO = finance_data_mean.USO.apply(lambda x : round(x, 2))
finance_data_mean

,USO
Date,
2025-10,71.12
2025-11,70.96
2025-12,69.48
2026-01,72.64
2026-02,78.54
2026-03,108.33


Store the dataset for further usage.

In [18]:
finance_data_mean.to_csv("../data/processed/finance_preprocessing.csv")